In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path("data/cement_calciner_cost_grid.csv")
FEATURE_COLUMNS = ['log10_co2_permeance_gpu', 'log10_co2_n2_selectivity', 'log10_co2_o2_selectivity']

df = pd.read_csv(DATA_PATH)
X = df[FEATURE_COLUMNS].to_numpy(dtype=float)
y = np.log10(df["optimized_cost"].to_numpy(dtype=float))

print(f"Loaded {len(df)} rows from {DATA_PATH}")
print("Features:", FEATURE_COLUMNS)

Loaded 81 rows from data/cement_calciner_cost_grid.csv
Features: ['log10_co2_permeance_gpu', 'log10_co2_n2_selectivity', 'log10_co2_o2_selectivity']


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error

RANDOM_STATE = 0

model = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf")),
])

param_grid = {
    "svr__C": np.logspace(-1, 4, 11),
    "svr__gamma": np.logspace(-3, 2, 11),
    "svr__epsilon": [0.03, 0.1],
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = GridSearchCV(
    model,
    param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=1,
)
search.fit(X, y)
best_model = search.best_estimator_

y_train_pred = best_model.predict(X)
y_cv_pred = cross_val_predict(best_model, X, y, cv=cv, n_jobs=-1)

train_r2 = r2_score(y, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y, y_train_pred))
cv_r2 = r2_score(y, y_cv_pred)
cv_rmse = np.sqrt(mean_squared_error(y, y_cv_pred))
train_rmse_cost = np.sqrt(mean_squared_error(10 ** y, 10 ** y_train_pred))
cv_rmse_cost = np.sqrt(mean_squared_error(10 ** y, 10 ** y_cv_pred))

print("\n===== Cement calciner surrogate =====")
best_params = {key: float(value) for key, value in search.best_params_.items()}
print("best_params:", best_params)
print(f"best_cv_r2 (mean over folds): {search.best_score_:.9f}")
print(f"train R2: {train_r2:.9f}")
print(f"train RMSE [log10(cost)]: {train_rmse:.9f}")
print(f"cv R2: {cv_r2:.9f}")
print(f"cv RMSE [log10(cost)]: {cv_rmse:.9f}")
print(f"train RMSE [USD2024/tCO2]: {train_rmse_cost:.9f}")
print(f"cv RMSE [USD2024/tCO2]: {cv_rmse_cost:.9f}")

Fitting 5 folds for each of 242 candidates, totalling 1210 fits



===== Cement calciner surrogate =====
best_params: {'svr__C': 31.622776601683793, 'svr__epsilon': 0.03, 'svr__gamma': 0.03162277660168379}
best_cv_r2 (mean over folds): 0.982632999
train R2: 0.990744125
train RMSE [log10(cost)]: 0.020068514
cv R2: 0.984065134
cv RMSE [log10(cost)]: 0.026331809
train RMSE [USD2024/tCO2]: 2.406324989
cv RMSE [USD2024/tCO2]: 3.173508498


In [3]:
QUERY_LABELS = ['log10 CO2 permeance', 'log10 CO2/N2 selectivity', 'log10 CO2/O2 selectivity']
x_query = np.array([[5.064, 1.285, 0.574]], dtype=float)

pred_log10_cost = best_model.predict(x_query)[0]
pred_cost = 10 ** pred_log10_cost

print("Example membrane performance")
for label, value in zip(QUERY_LABELS, x_query[0]):
    print(f"{label}: {value}")
print(f"Predicted log10(cost): {pred_log10_cost:.6f}")
print(f"Predicted cost: {pred_cost:.6f} USD2024/tCO2")

Example membrane performance
log10 CO2 permeance: 5.064
log10 CO2/N2 selectivity: 1.285
log10 CO2/O2 selectivity: 0.574
Predicted log10(cost): 1.717815
Predicted cost: 52.217353 USD2024/tCO2
